# AI Agents with Google Gemini (Free API)

## What is an AI Agent?

An **AI Agent** is an autonomous system that can:
- **Perceive** its environment through inputs
- **Reason** about what actions to take
- **Act** by executing tasks and using tools
- **Learn** from feedback to improve performance

In this notebook, we'll build AI agents using **Google Gemini's FREE API**!

### Why Gemini for AI Agents?

✅ **FREE API**: No credit card needed

✅ **Function Calling**: Native support for tool use

✅ **Fast**: Good latency for agent loops

✅ **Reliable**: Production-grade API

### Key Components of AI Agents:
1. **LLM Brain**: Gemini for reasoning and decision-making
2. **Tools**: Functions the agent can call (APIs, calculators, etc.)
3. **Memory**: Conversation context and history
4. **Planning**: Breaking down complex tasks
5. **Reflection**: Self-evaluation and error correction

---

## Setup and Installation

In [ ]:
# Install required packages
!pip install -q google-generativeai requests wikipedia-api

In [ ]:
import os
import json
import requests
from datetime import datetime
from typing import List, Dict, Callable, Any
import time

import google.generativeai as genai

print("✅ Packages imported successfully!")

In [ ]:
# Configure API key
GOOGLE_API_KEY = "YOUR_API_KEY_HERE"  # Replace with your key

# Or use environment variable
# GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')

genai.configure(api_key=GOOGLE_API_KEY)

print("✅ Gemini API configured!")
print("\n🎓 Get your free API key at: https://makersuite.google.com/app/apikey")

---
## Building a ReAct Agent with Gemini

We'll implement a **ReAct (Reasoning + Acting)** agent that can:
- Use multiple tools
- Reason about which tool to use
- Execute actions
- Provide answers based on tool outputs

### Step 1: Define Tools

Tools are Python functions that the agent can call.

In [ ]:
# Tool 1: Calculator
def calculator(expression: str) -> str:
    """Evaluates a mathematical expression."""
    try:
        result = eval(expression)
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {str(e)}"

# Tool 2: Wikipedia Search
def wikipedia_search(query: str) -> str:
    """Searches Wikipedia for information about a topic."""
    try:
        import wikipediaapi
        wiki = wikipediaapi.Wikipedia(
            user_agent='StudentLearningAgent/1.0',
            language='en'
        )
        page = wiki.page(query)
        if page.exists():
            # Return first 500 characters
            return page.summary[:500] + "..."
        else:
            return f"No Wikipedia page found for '{query}'"
    except Exception as e:
        return f"Error searching Wikipedia: {str(e)}"

# Tool 3: Current Date/Time
def get_current_datetime() -> str:
    """Returns the current date and time."""
    now = datetime.now()
    return now.strftime("%Y-%m-%d %H:%M:%S")

# Tool 4: Weather Information (Mock)
def get_weather(city: str) -> str:
    """Gets weather information for a city (mock data)."""
    # In a real implementation, you'd call a weather API
    weather_data = {
        "london": "Cloudy, 15°C",
        "new york": "Sunny, 22°C",
        "tokyo": "Rainy, 18°C",
        "paris": "Partly cloudy, 17°C",
        "rome": "Sunny, 24°C",
        "berlin": "Overcast, 13°C"
    }
    city_lower = city.lower()
    return weather_data.get(city_lower, f"Weather data not available for {city}")

print("✅ Tools defined successfully")

### Step 2: Register Tools with Gemini

Gemini needs to know about our tools in a specific format.

In [ ]:
# Define tool declarations for Gemini
calculator_tool = genai.protos.Tool(
    function_declarations=[
        genai.protos.FunctionDeclaration(
            name='calculator',
            description='Evaluates mathematical expressions. Use for calculations.',
            parameters=genai.protos.Schema(
                type=genai.protos.Type.OBJECT,
                properties={
                    'expression': genai.protos.Schema(
                        type=genai.protos.Type.STRING,
                        description='The mathematical expression to evaluate, e.g., "2+2" or "10*5"'
                    )
                },
                required=['expression']
            )
        ),
        genai.protos.FunctionDeclaration(
            name='wikipedia_search',
            description='Searches Wikipedia for information. Use when you need factual information.',
            parameters=genai.protos.Schema(
                type=genai.protos.Type.OBJECT,
                properties={
                    'query': genai.protos.Schema(
                        type=genai.protos.Type.STRING,
                        description='The topic to search for on Wikipedia'
                    )
                },
                required=['query']
            )
        ),
        genai.protos.FunctionDeclaration(
            name='get_current_datetime',
            description='Returns the current date and time. Use when asked about the current date or time.',
            parameters=genai.protos.Schema(
                type=genai.protos.Type.OBJECT,
                properties={}
            )
        ),
        genai.protos.FunctionDeclaration(
            name='get_weather',
            description='Gets current weather information for a city.',
            parameters=genai.protos.Schema(
                type=genai.protos.Type.OBJECT,
                properties={
                    'city': genai.protos.Schema(
                        type=genai.protos.Type.STRING,
                        description='The name of the city'
                    )
                },
                required=['city']
            )
        )
    ]
)

# Map function names to actual functions
AVAILABLE_FUNCTIONS = {
    'calculator': calculator,
    'wikipedia_search': wikipedia_search,
    'get_current_datetime': get_current_datetime,
    'get_weather': get_weather
}

print(f"✅ Registered {len(AVAILABLE_FUNCTIONS)} tools with Gemini")

### Step 3: Implement the Agent

The agent follows a ReAct loop:
1. **Think**: Reason about what to do
2. **Act**: Call a tool
3. **Observe**: Get the result
4. **Repeat** until the task is complete

In [ ]:
class GeminiAgent:
    """A ReAct agent powered by Gemini that can use tools."""
    
    def __init__(self, max_iterations: int = 5):
        self.max_iterations = max_iterations
        self.model = genai.GenerativeModel(
            'gemini-pro',
            tools=[calculator_tool]
        )
        self.chat = None
    
    def run(self, user_query: str, verbose: bool = True) -> str:
        """Run the agent on a user query."""
        # Initialize chat with system instruction
        self.chat = self.model.start_chat(enable_automatic_function_calling=False)
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"🤖 Gemini Agent Started")
            print(f"📝 Query: {user_query}")
            print(f"{'='*70}\n")
        
        # Send initial query
        response = self.chat.send_message(user_query)
        
        # ReAct loop
        for iteration in range(self.max_iterations):
            if verbose:
                print(f"\n--- Iteration {iteration + 1} ---")
            
            # Check if model wants to call a function
            if response.candidates[0].content.parts[0].function_call:
                function_call = response.candidates[0].content.parts[0].function_call
                function_name = function_call.name
                function_args = dict(function_call.args)
                
                if verbose:
                    print(f"\n🔧 Calling tool: {function_name}")
                    print(f"   Arguments: {function_args}")
                
                # Execute the function
                if function_name in AVAILABLE_FUNCTIONS:
                    function_to_call = AVAILABLE_FUNCTIONS[function_name]
                    function_response = function_to_call(**function_args)
                    
                    if verbose:
                        print(f"   Result: {function_response}")
                    
                    # Send function response back to model
                    response = self.chat.send_message(
                        genai.protos.Content(
                            parts=[genai.protos.Part(
                                function_response=genai.protos.FunctionResponse(
                                    name=function_name,
                                    response={'result': function_response}
                                )
                            )]
                        )
                    )
                else:
                    if verbose:
                        print(f"   ⚠️  Unknown function: {function_name}")
                    break
            else:
                # No more function calls, return final answer
                final_answer = response.text
                if verbose:
                    print(f"\n{'='*70}")
                    print(f"✅ Final Answer:\n{final_answer}")
                    print(f"{'='*70}\n")
                return final_answer
        
        return "Max iterations reached without final answer."

print("✅ GeminiAgent class created")

---
## Testing the Agent

Let's test our agent with various queries that require different tools.

### Test 1: Mathematical Calculation

In [ ]:
agent = GeminiAgent()
answer = agent.run("What is 234 * 567?")

### Test 2: Information Retrieval

In [ ]:
answer = agent.run("Tell me about the Eiffel Tower")
time.sleep(1)  # Rate limiting

### Test 3: Current Information

In [ ]:
answer = agent.run("What's the date today?")
time.sleep(1)

### Test 4: Multi-Step Reasoning

In [ ]:
answer = agent.run("What's the weather in Paris, and tell me one famous landmark there?")
time.sleep(1)

### Test 5: Complex Query

In [ ]:
answer = agent.run(
    "If I have 150 euros and the Eiffel Tower ticket costs 28.30 euros, "
    "how many tickets can I buy? Also, what's the weather like in Paris?"
)
time.sleep(1)

---
## Agent with Memory

Let's create an agent that remembers previous interactions.

In [ ]:
class GeminiAgentWithMemory(GeminiAgent):
    """Agent with conversation memory."""
    
    def __init__(self, max_iterations: int = 5):
        super().__init__(max_iterations)
        # Initialize chat once for persistent memory
        self.chat = self.model.start_chat(enable_automatic_function_calling=False)
    
    def run(self, user_query: str, verbose: bool = True) -> str:
        """Run the agent with memory of previous interactions."""
        if verbose:
            print(f"\n{'='*70}")
            print(f"🤖 Gemini Agent with Memory")
            print(f"📝 Query: {user_query}")
            print(f"{'='*70}\n")
        
        # Send query (chat remembers history)
        response = self.chat.send_message(user_query)
        
        # ReAct loop
        for iteration in range(self.max_iterations):
            if verbose:
                print(f"\n--- Iteration {iteration + 1} ---")
            
            # Check for function calls
            if response.candidates[0].content.parts[0].function_call:
                function_call = response.candidates[0].content.parts[0].function_call
                function_name = function_call.name
                function_args = dict(function_call.args)
                
                if verbose:
                    print(f"\n🔧 Tool: {function_name}({function_args})")
                
                # Execute function
                if function_name in AVAILABLE_FUNCTIONS:
                    function_to_call = AVAILABLE_FUNCTIONS[function_name]
                    function_response = function_to_call(**function_args)
                    
                    if verbose:
                        print(f"   Result: {function_response}")
                    
                    # Send response back
                    response = self.chat.send_message(
                        genai.protos.Content(
                            parts=[genai.protos.Part(
                                function_response=genai.protos.FunctionResponse(
                                    name=function_name,
                                    response={'result': function_response}
                                )
                            )]
                        )
                    )
                else:
                    break
            else:
                # Final answer
                final_answer = response.text
                if verbose:
                    print(f"\n{'='*70}")
                    print(f"✅ Answer: {final_answer}")
                    print(f"{'='*70}\n")
                return final_answer
        
        return "Max iterations reached."

print("✅ Agent with memory created")

### Testing Agent with Memory

In [ ]:
# Create agent with memory
memory_agent = GeminiAgentWithMemory()

# First query
memory_agent.run("What's 25 * 4?")
time.sleep(1)

In [ ]:
# Second query - agent remembers the previous calculation
memory_agent.run("Now add 50 to that result")
time.sleep(1)

In [ ]:
# Third query - testing contextual memory
memory_agent.run("What was my first question?")
time.sleep(1)

---
## Agent Design Patterns

### 1. ReAct Pattern (What we implemented)
- **Reason** → **Act** → **Observe** loop
- Best for: General-purpose tasks, exploratory problems

### 2. Plan-and-Execute Pattern
- Create a plan first
- Execute steps sequentially
- Best for: Complex multi-step tasks

### 3. Reflexion Pattern
- Execute → Evaluate → Refine
- Agent reflects on its mistakes
- Best for: Tasks requiring high accuracy

### 4. Multi-Agent Pattern
- Multiple specialized agents
- Coordinate and communicate
- Best for: Complex systems, different expertise domains

---
## Adding Custom Tools

You can easily add more tools to your agent!

In [ ]:
# Example: Currency converter tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Converts currency (mock exchange rates)."""
    # Mock exchange rates (USD base)
    rates = {
        'USD': 1.0,
        'EUR': 0.85,
        'GBP': 0.73,
        'JPY': 110.0
    }
    
    from_currency = from_currency.upper()
    to_currency = to_currency.upper()
    
    if from_currency not in rates or to_currency not in rates:
        return f"Currency not supported. Supported: {list(rates.keys())}"
    
    # Convert to USD first, then to target currency
    usd_amount = amount / rates[from_currency]
    result = usd_amount * rates[to_currency]
    
    return f"{amount} {from_currency} = {result:.2f} {to_currency}"

# Test it
print(convert_currency(100, 'USD', 'EUR'))
print("\n💡 To use this in your agent, add it to the tool declarations!")

---
## Best Practices for AI Agents

### 1. Tool Design
- ✅ Keep tools focused and single-purpose
- ✅ Provide clear descriptions for the LLM
- ✅ Handle errors gracefully with try/except
- ✅ Return string outputs for consistency

### 2. Agent Configuration
- ✅ Set reasonable iteration limits (3-5)
- ✅ Add verbose mode for debugging
- ✅ Implement rate limiting (especially with free tier)
- ✅ Handle API errors gracefully

### 3. Cost Management with Free Tier
- ✅ Gemini Free: 60 requests/minute, 1,500/day
- ✅ Add `time.sleep(1)` between agent runs
- ✅ Cache results when possible
- ✅ Minimize unnecessary tool calls

### 4. Testing
- ✅ Test each tool independently first
- ✅ Start with simple queries
- ✅ Gradually increase complexity
- ✅ Monitor tool call patterns

---
## Real-World Applications

### Customer Support Agent
- Access to knowledge base
- Can check order status
- Can create support tickets
- Can escalate to human

### Research Assistant Agent
- Search academic papers
- Summarize findings
- Track citations
- Generate reports

### Personal Assistant Agent
- Check calendar
- Send emails
- Set reminders
- Manage tasks

### Data Analysis Agent
- Query databases
- Generate visualizations
- Run statistical tests
- Create reports

---
## Limitations & Challenges

### Current Limitations:
1. **Tool Calling Reliability**: LLM might not always choose the right tool
2. **Error Recovery**: Agents can get stuck in loops
3. **Cost**: Can add up with many iterations (use free Gemini!)
4. **Latency**: Each iteration adds delay
5. **Safety**: Agents can execute arbitrary code (be careful!)

### Mitigation Strategies:
- Set maximum iteration limits
- Provide clear tool descriptions
- Implement fallback mechanisms
- Add human-in-the-loop for critical actions
- Sandbox tool execution
- Use Gemini free tier for learning

---
## Exercise: Build Your Own Agent

Try creating an agent with these tools:

1. **Email sender** (mock)
2. **Calendar checker** (mock)
3. **Task manager** (create/list/complete tasks)

The agent should help with:
- Scheduling meetings
- Sending reminders
- Managing priorities

In [ ]:
# Your code here!
# Hint: Follow the same pattern we used above
#
# 1. Define your tool functions
# def send_email(to: str, subject: str, body: str) -> str:
#     # Your implementation
#     pass
#
# 2. Create tool declarations for Gemini
# 3. Create your agent
# 4. Test with queries

---
## Summary

### What We Learned

1. ✅ **Built AI Agents with Gemini** (100% FREE!)
2. ✅ **Implemented ReAct Pattern** (Reason → Act → Observe)
3. ✅ **Created Multiple Tools** (calculator, Wikipedia, weather, datetime)
4. ✅ **Added Memory** for multi-turn conversations
5. ✅ **Tested Complex Scenarios** requiring multiple tools

### Key Takeaways

📌 **Agents = LLM + Tools + Loop**
- LLM (Gemini) for reasoning
- Tools for taking actions
- Loop for iterative problem solving

📌 **Gemini is Perfect for Students**
- FREE API with generous limits
- Native function calling support
- Good quality and reliability

📌 **Design Patterns Matter**
- ReAct: general purpose
- Plan-Execute: complex tasks
- Reflexion: high accuracy
- Multi-agent: specialized domains

📌 **Best Practices**
- Clear tool descriptions
- Error handling
- Rate limiting
- Maximum iterations

### Next Steps

1. **Experiment**: Add more tools to your agent
2. **Combine**: Use agents with RAG for knowledge-grounded responses
3. **Scale**: Build multi-agent systems
4. **Deploy**: Create a web interface with Streamlit
5. **Learn**: Explore LangChain and other agent frameworks

---

### Resources

- [Google AI Studio](https://makersuite.google.com/)
- [Gemini Function Calling Docs](https://ai.google.dev/docs/function_calling)
- [LangChain Documentation](https://python.langchain.com/)
- [ReAct Paper (Yao et al., 2023)](https://arxiv.org/abs/2210.03629)

🎉 **You're now ready to build AI agents with Gemini!**